In [18]:
# Cell 1: Imports and Worker Definition
import time
from multiprocessing import get_context, current_process

In [22]:
def heavy_compute(x):
    """Simulate a CPU-bound task by summing squares up to x."""
    proc = current_process().name
    print(f"[{proc}] Computing sum of squares up to {x}\n")
    total = sum(i * i for i in range(x))
    print(f"[{proc}] Done computing {x}")
    return total

In [20]:
# Cell 2: Launching Explicit Processes
ctx = get_context('fork')  # Ensures fork() start method in Jupyter

# Define workloads
numbers = [5_000_000, 5_000_000]
processes = []

# Create and start processes
for idx, n in enumerate(numbers, start=1):
    p = ctx.Process(target=heavy_compute, args=(n,), name=f"Worker-{idx}")
    processes.append(p)
    p.start()

# Wait for all to finish
for p in processes:
    p.join()

print("All explicit processes have completed.")

[Worker-1] Computing sum of squares up to 5000000
[Worker-2] Computing sum of squares up to 5000000
[Worker-1] Done computing 5000000
[Worker-2] Done computing 5000000
All explicit processes have completed.


In [23]:
ctx = get_context('fork')  # Use fork in notebooks
inputs = [2_000_000, 3_000_000, 4_000_000, 5_000_000]

# Create a pool of 4 worker processes
with ctx.Pool(processes=4) as pool:
    results = pool.map(heavy_compute, inputs)

print("Pool results:", results)

[ForkPoolWorker-52] Computing sum of squares up to 4000000
[ForkPoolWorker-51] Computing sum of squares up to 3000000
[ForkPoolWorker-50] Computing sum of squares up to 2000000
[ForkPoolWorker-53] Computing sum of squares up to 5000000




[ForkPoolWorker-50] Done computing 2000000
[ForkPoolWorker-51] Done computing 3000000
[ForkPoolWorker-52] Done computing 4000000
[ForkPoolWorker-53] Done computing 5000000
Pool results: [2666664666667000000, 8999995500000500000, 21333325333334000000, 41666654166667500000]


In [5]:
import time
import threading
from multiprocessing import get_context

In [7]:
def io_task(name: str, duration: float):
    """
    Simulate an I/O-bound operation by sleeping.
    """
    print(f"[{time.strftime('%X')}] IO task {name}: starting (sleep {duration}s)")
    time.sleep(duration)
    print(f"[{time.strftime('%X')}] IO task {name}: finished\n")

In [3]:
def cpu_task(name: str, n: int):
    """
    Simulate a CPU-bound operation by computing a sum of squares.
    """
    print(f"[{time.strftime('%X')}] CPU task {name}: starting (n={n})")
    start_time = time.time()
    total = sum(i * i for i in range(n))
    elapsed = time.time() - start_time
    print(f"[{time.strftime('%X')}] CPU task {name}: finished in {elapsed:.2f}s")
    return total

In [8]:

# --- Concurrency with threading ---
print("\n=== Concurrency (I/O-bound) ===")
durations = [2, 2]
threads = [
    threading.Thread(target=io_task, args=(f"T{i}", d))
    for i, d in enumerate(durations, 1)
]
for t in threads: t.start()
for t in threads: t.join()

# --- Parallelism with multiprocessing (force fork) ---
print("\n=== Parallelism (CPU-bound) ===")
N = 5_000_000
ctx = get_context("fork")           # <-- force fork instead of spawn for notebook
with ctx.Pool(2) as pool:
        pool.starmap(cpu_task, [(f"P1", N), (f"P2", N)])


=== Concurrency (I/O-bound) ===
[14:05:13] IO task T1: starting (sleep 2s)
[14:05:13] IO task T2: starting (sleep 2s)
[14:05:15] IO task T1: finished
[14:05:15] IO task T2: finished



=== Parallelism (CPU-bound) ===
[14:05:15] CPU task P1: starting (n=5000000)[14:05:15] CPU task P2: starting (n=5000000)

[14:05:15] CPU task P2: finished in 0.38s
[14:05:15] CPU task P1: finished in 0.38s
